# 피벗테이블과 다중 집계 함수
- `pivot_table()`로 두 개의 범주형 변수(지역 x 카테고리)를 축으로 하는 교차표
- 하나의 피벗테이블 안에서 평균/합계/개수 등 여러 집계 함수를 동시에 적용

### 💡 강의 포인트
- 예제08의 groupby+unstack과 결과는 비슷해 보이지만, `pivot_table`은 `aggfunc`에 리스트/딕셔너리를 넘겨 **여러 통계를 한 번에** 뽑을 수 있다는 게 핵심 차이 — "groupby+unstack의 상위호환"이라고 설명하면 이해가 빠름.
- `margins=True` 옵션으로 행/열 합계(총계) 줄을 자동으로 붙일 수 있다는 것도 실무에서 자주 쓰이는 포인트로 짚어주기.
- 결측 조합(예: 특정 지역에 특정 카테고리 주문이 아예 없는 경우) 처리를 위해 `fill_value=0`을 같이 설명하면 좋음.

In [ ]:
import os
from dotenv import load_dotenv
import oracledb
import pandas as pd

load_dotenv() 

USER = os.getenv("ORACLE_USER")
PASSWORD = os.getenv("ORACLE_PASSWORD")
DSN = os.getenv("ORACLE_DSN")


In [ ]:
with oracledb.connect(user=USER, password=PASSWORD, dsn=DSN) as conn:
    df = pd.read_sql("SELECT * FROM DELIVERY_ORDERS", conn)


df.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rc('font', family='Malgun Gothic') 
mpl.rc('axes', unicode_minus=False)


In [ ]:
import numpy as np

pivot_price = pd.pivot_table(
    df,
    values="PRICE",
    index="REGION",
    columns="CATEGORY",
    aggfunc="mean",
    fill_value=0,
    margins=True,
    margins_name="전체평균",
)
print("=== 지역 x 카테고리 평균 가격 ===")
print(pivot_price.round(0))

In [ ]:
pivot_multi = pd.pivot_table(
    df,
    values="PRICE",
    index="REGION",
    columns="CATEGORY",
    aggfunc=["mean", "count"],
    fill_value=0,
)
print("=== 평균 가격 + 주문 건수 동시 집계 ===")
print(pivot_multi)

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.heatmap(pivot_price.drop("전체평균", axis=0).drop("전체평균", axis=1),
            annot=True, fmt=".0f", cmap="YlOrRd")
plt.title("지역 x 카테고리 평균 가격 히트맵")
plt.show()

### ❓ 생각해볼 질문
1. `pivot_table`과 예제02(groupby+unstack)의 결과가 어떤 상황에서 서로 달라질 수 있을까? (결측 조합 처리 방식 차이)
2. `aggfunc=["mean", "count"]`처럼 리스트로 넘기지 않고 컬럼마다 다른 통계를 적용하려면 어떻게 해야 할까? (딕셔너리 형태 힌트: `{"PRICE": "mean", "RATING": "max"}`)